In [10]:
import os
import pandas as pd
import subprocess, pathlib
import json

TYPE_JOURNAL = "journal"
TYPE_POSITION = "position"
TYPE_PRESENTATION = "presentation"

txt = "/Users/chuansun/Downloads/input.txt"
f = open(txt, "r")
raw_contents = f.readlines()

In [11]:

raw_contents[1].replace("\n", "")

'Brussow, J.A. (2017, December). Considerations when interpreting scores from multiple test attempts. Leawood, KS: ATI Nursing.'

In [12]:
my_env = os.environ
my_env["GEM_HOME"] = "/Users/chuansun/bin/gems/"
process = subprocess.Popen(["/Library/Ruby/Gems/2.3.0/gems/anystyle-cli-1.2.0/bin/anystyle","-f","json","parse", txt],
                     env=my_env,
                     stdout=subprocess.PIPE, 
                     stderr=subprocess.PIPE)
stdout, stderr = process.communicate()
stdout, stderr

(b'[\n  {\n    "author": [\n      {\n        "family": "Brussow",\n        "given": "J.A."\n      },\n      {\n        "family": "Dunham",\n        "given": "M."\n      }\n    ],\n    "date": [\n      "2018"\n    ],\n    "title": [\n      "Students\xe2\x80\x99 mid-program content area performance as a predictor of end-of-program NCLEX readiness"\n    ],\n    "volume": [\n      "43"\n    ],\n    "type": "article-journal",\n    "container-title": [\n      "Nurse Educator"\n    ],\n    "issue": [\n      "5"\n    ]\n  },\n  {\n    "author": [\n      {\n        "family": "Brussow",\n        "given": "J.A."\n      }\n    ],\n    "date": [\n      "2017-12"\n    ],\n    "title": [\n      "Considerations when interpreting scores from multiple test attempts"\n    ],\n    "location": [\n      "Leawood, KS"\n    ],\n    "publisher": [\n      "ATI Nursing"\n    ],\n    "type": "book"\n  },\n  {\n    "author": [\n      {\n        "family": "Liu",\n        "given": "X."\n      },\n      {\n        "f

In [14]:
TYPE = TYPE_JOURNAL

def extract_key(obj, name):
    if name in obj:
        return obj[name][0]
    return None

raw_json = json.loads(stdout.decode("utf-8").replace("\n", ""))
results = []
for idx, item in enumerate(raw_json):
    # author
    _author_list = []
    for _author in item["author"]:
        _author_list.append(f"{_author['family']}, {_author['given']}")
    _author_str = " & ".join(_author_list)
    
    # date
    _date = item["date"][0]
    _date_split = _date.split("-")
    if len(_date_split) > 1:
        _year, _month = _date_split
    else:
        _year, _month = _date, None
        
    _title = item["title"][0]
    _type = TYPE
    if _type == TYPE_JOURNAL:
        _journal_name = extract_key(item, "container-title")
        if "issue" not in item:
            _volume_issue = None
            _journal_page = None
        else: 
            _volume_issue = f"{extract_key(item, 'volume')}({extract_key(item, 'issue')})"
            _journal_page = extract_key(item, "page")
    else:
        _journal_name = None
        _volume_issue = None
        _journal_page = None
    _location = extract_key(item, "location")
    
    if _type == TYPE_PRESENTATION:
        _conference = item["container-title"][0]
    else:
        _conference = None
        
    _raw = raw_contents[idx].replace("\n", "").replace("’", "'")
    row = [_author_str, _year, _month, _title, _type, _journal_name, _volume_issue, _journal_page, _location, _conference, _raw]
    results.append(row)

df = pd.DataFrame(results, columns=["author","year","month","title","type","journal_name","volume_issue",
                               "journal_page","location","conference","raw"])
df

,author,year,month,title,type,journal_name,volume_issue,journal_page,location,conference,raw
0,"Brussow, J.A. & Dunham, M.",2018,None,Students’ mid-program content area performance...,journal,Nurse Educator,43(5),None,None,None,"Brussow, J.A. & Dunham, M. (2018). Students' m..."
1,"Brussow, J.A.",2017,12,Considerations when interpreting scores from m...,journal,None,None,None,"Leawood, KS",None,"Brussow, J.A. (2017, December). Considerations..."
2,"Liu, X. & Gage, A. & Codd, C. & Mills, C.",2018,02,The use of resampling techniques for improving...,journal,Presentation at the Innovations in Testing Ann...,None,None,"San Antonio, TX",None,"Liu, X., Gage, A., Codd, C., & Mills, C. (2018..."


In [15]:
df.to_csv("~/Downloads/publication.txt", sep="*")

In [58]:
item = raw_json[1]
_location = None if "location" not in item else item["location"][0]
_location'’

'Leawood, KS'